### Byte-Pair Encoding

In this notebook, you'll implement the byte-pair encoding algorithm to build a tokenizer.

In [ ]:
from collections import Counter

We'll build this on a small piece of text. Once you get the code working, you can try it out on a larger corpus and/or try more iterations.

In [ ]:
text = """
Did you ever hear the tragedy of Darth Plagueis The Wise? I thought not. It’s not a story the Jedi would tell you. It’s a Sith legend. Darth Plagueis was a Dark Lord of the Sith, so powerful and so wise he could use the Force to influence the midichlorians to create life… He had such a knowledge of the dark side that he could even keep the ones he cared about from dying. The dark side of the Force is a pathway to many abilities some consider to be unnatural. He became so powerful… the only thing he was afraid of was losing his power, which eventually, of course, he did. Unfortunately, he taught his apprentice everything he knew, then his apprentice killed him in his sleep. Ironic. He could save others from death, but not himself.
"""

print(text)

We'll start by preprocessing the text, lowercasing all text and splitting on spaces.

In [ ]:
words = text.lower().split()
print(f"Total Words: {len(words)}")

merge_rules = []

**Part 1:** Create a function, tokenize_word that takes as input as word and outputs that word converted to a tuple of characters plus an end of word character, `</w>`.

**Example Input**: `'darth'`  
**Example Output:** `('d', 'a', 'r', 't', 'h', '</w>')`.

In [ ]:
# Your Code Here

Now, we can apply this function across each word.

In [ ]:
tokenized = [tokenize_word(word) for word in words]
tokenized[:5]

We'll take the result of this and build a vocabulary counter.

In [ ]:
vocab = Counter(tokenized)
vocab

**Part 2:** Create a function, create_pairs, that takes as input a tuple of strings and which returns a list containing all tuples that can be created by taking pairs on consecutive elements of the input tuple.

**Example Input:** `('n', 'o', 't', '</w>')`  
**Example Output:** `[('n', 'o'), ('o', 't'), ('t', '</w>')]`

In [ ]:
# Your Code Here

In [ ]:
create_pairs(('n', 'o', 't', '</w>'))

Now, we'll use the function along with the vocabulary frequencies to count the most common pairs.

In [ ]:
pairs = []
for word, freq in vocab.items():
    pairs.extend(create_pairs(word) * freq)

pairs = Counter(pairs)
pairs.most_common(1)

**Part 3:** Now, we need to merge the most common pair.

To do this, you may want to create a function that accepts a pair_to_merge and a word tuple and returns the word tuple after merging any instance of the pair_to_merge.

**Example Input:** word = `('t', 'h', 'e', '</w>')`, pair_to_merge =  `('e', '</w>')`  
**Example Output:** `('t', 'h', 'e</w>')`

**Example Input:** word = `('d', 'a', 'r', 't', 'h', '</w>')`, pair_to_merge =  `('e', '</w>')`  
**Example Output:** `('d', 'a', 'r', 't', 'h', '</w>')`

In [ ]:
# Your Code Here

In [ ]:
merge_pair(word = ('t', 'h', 'e', '</w>'), pair_to_merge = ('e', '</w>'))

In [ ]:
merge_pair(word = ('d', 'a', 'r', 't', 'h', '</w>'), pair_to_merge = ('e', '</w>'))

Now, we can apply this function across the vocabulary to create our new vocabulary.

In [ ]:
new_vocab = {}
new_merge = pairs.most_common()[0][0]
merge_rules.append(new_merge)

for word in vocab:
    new_vocab[merge_pair(word, new_merge)] = vocab[word]

vocab = new_vocab

**Part 4:** Write a for loop so that you can repeat the process of finding the most common pair and then updating the vocabulary. Run this for 10 iterations.

In [ ]:
# Your Code Here

Let's see the list of new tokens after the 11 iterations we have performed.

In [ ]:
tokens = []
for word in vocab:
    tokens.extend(word)
tokens = list(set(tokens))
sorted(tokens)

And here's our set of merge rules.

In [ ]:
merge_rules

If we want to tokenize some new text, this is what it would look like. Note that we have to merge in the same order as we did with the original text.

In [ ]:
new_text = "It's over Anakin, I have the high ground."
new_words = new_text.lower().split()

new_tokenized = [tokenize_word(word) for word in new_words]
new_tokenized

And then a helper function to apply byte-pair encoding using our rules.

In [ ]:
def apply_bpe(word, merge_rules):
    word = list(word)
    while True:
        pairs = [(word[i], word[i+1]) for i in range(len(word)-1)]
        merge_found = False
        for merge in merge_rules:
            if merge in pairs:
                i = pairs.index(merge)
                word = word[:i] + [''.join(merge)] + word[i+2:]
                merge_found = True
                break
        if not merge_found:
            break
    return tuple(word)

In [ ]:
[apply_bpe(word, merge_rules) for word in new_tokenized]

**Bonus Challenge: Training a Larger Byte-Pair Encoding Tokenizer**

In the previous notebook, you worked with the top 50 Project Gutenberg books and created tokenized representations of each document. Instead of training byte-pair encoding on a single paragraph, let's see what happens when we train it on a much larger corpus.

Combine the text from all 50 Project Gutenberg books into a single large string and train your byte-pair encoding tokenizer on this larger corpus.

You may want to:
* reuse your import_book function from the previous notebook,
* iterate through all files using glob,
* concatenate the books together into one string.

Train your tokenizer using different numbers of merge operations such as:
* 10 merges
* 50 merges
* 100 merges

After training, investigate the learned tokens and answer the following questions:
* What kinds of subwords or patterns are learned?
* Do common suffixes or prefixes appear as tokens?
* Which tokens appear to represent full words?
* How does increasing the number of merges change the vocabulary?

**Additional Stretch Goal: Optimizing Byte-Pair Encoding**
The implementation of byte-pair encoding above recomputes pair frequencies across the entire vocabulary during every merge iteration.

This works well for small corpora, but can become slow when training on large collections of text such as the full Project Gutenberg corpus.

Modify your implementation so that it updates pair counts more efficiently.

Instead of recomputing all pairs after every merge:
* Keep track of which pairs occur inside each word.
* Create a reverse lookup that maps each pair to the words that contain it.
* After merging a pair, only update the words affected by that merge.

You may find the following data structures useful:

`word_to_pairs[word]`

Stores the list of consecutive pairs contained in a word.

**Example:**

`('t', 'h', 'e', '</w>')`

might map to:

`[('t', 'h'), ('h', 'e'), ('e', '</w>')]`

---

`pair_to_words[pair]`

Stores the set of words containing a particular pair.

Example:

`('t', 'h')`

might map to:

`{
    ('t', 'h', 'e', '</w>'),
    ('a', 'n', 't', 'h', 'e', 'm', '</w>')
}`


Then try running 1000 iterations or more on the full corpus and inspecting the results.